# Exercise 5.2 — Hyperparameters: choosing the lasso and ridge penalty

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-02-regularisation.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

Some settings of a model are not learnt from the data: we choose them before
fitting. They are called **hyperparameters**. Here they are the two settings
of the penalty: its size, $\lambda$, and the mix of lasso and ridge (see
[Lasso and ridge in a picture](day5-01b-lasso-ridge-picture.qmd)). We choose
them by trying several values and keeping the ones that predict best on rows
the model was not fitted on.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit, ValidationCurveDisplay
from sklearn.metrics import root_mean_squared_error

The same 750 people and the same split as Exercise 5.1, from
[`data/diabetes_clean.csv`](data/diabetes_clean.csv). The outcome is glucose
again, but this time the other seven measurements are the predictors. In
`pressure`, `triceps` and `insulin` a zero means "not measured", so we turn the
zeros into missing values and let the pipeline fill them in. As in Exercise
5.1, 80% of the people are used for training and 20% are held back as the test
set; `random_state=42` makes the random split the same every time.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv").drop(columns="diabetes")   # glucose is the outcome
for col in ["pressure", "triceps", "insulin"]:
    diabetes[col] = diabetes[col].replace(0, np.nan)                         # 0 means "not measured"

predictors = ["pregnant", "pressure", "triceps", "insulin", "mass", "pedigree", "age"]

train, test = train_test_split(diabetes, test_size=0.2, random_state=42)

------------------------------------------------------------------------

### Part 1 — A model with room to overfit

The pipeline fills in the missing values with the median (Day 2), adds every
square and every product of two predictors with
`PolynomialFeatures(degree=2)`, and standardises the result. Seven predictors
become 35 columns: plenty of room to fit noise.

The model is the lasso, `Lasso()`. Its $\lambda$ is called `alpha`.
(`max_iter` lets the fitting run longer than the default, because the squares
and products are correlated and take longer to fit.)

In [ ]:
lasso_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),                # fill the missing values
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),  # squares and products
    ("scale", StandardScaler()),                                 # one scale for every column
    ("model", Lasso(max_iter=100000)),
])

The hyperparameter of a step in a pipeline is set with `set_params()`, using
the step's name, two underscores and the setting's name: `model__alpha`.

------------------------------------------------------------------------

### Part 2 — A grid search, scored on the training rows

A **grid search** tries a list of values, fits one model for each, and keeps
the one with the lowest error. Try seven penalties, each about three times the
one before. Fit each model on the training rows and, for now, measure its RMSE
on the **same** rows:

In [ ]:
# Replace every ______ below, then run this cell.
penalties = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
rows = []

for p in penalties:
    lasso_pipe.set_params(model__alpha=p)                  # fill in the hyperparameter
    lasso_pipe.fit(train[predictors], train["glucose"])

    train_rmse = root_mean_squared_error(______["glucose"], lasso_pipe.predict(______[predictors]))
    rows.append({"penalty": p, "train_rmse": train_rmse})

train_results = pd.DataFrame(rows)
train_results.round(2)

> **Stuck?**
>
> For now, the model is scored on the rows it was fitted on.

The RMSE only ever goes up: 26.83 at a penalty of 0.01, 30.66 at 10. Scored on
its own training rows, the smallest penalty always wins, and no penalty at all
would win outright.

**Question!** Does that mean the penalty is useless?

------------------------------------------------------------------------

### Part 3 — The same grid search, scored on a validation set

The training RMSE cannot show overfitting: the noise the model has learnt is in
the very rows it is scored on. We need rows it has **not** seen. Not the test
rows: they are kept for one final check, and choosing with them would make that
check too optimistic (the Stretch of Exercise 5.1). So we split the training
rows once more, with the same function as before: a random quarter of them
become **validation** rows.

In [ ]:
fitting, validation = train_test_split(train, test_size=0.25, random_state=42)   # 75% to fit, 25% to compare

print(len(fitting), len(validation))

Run the grid search again: fit on `fitting`, score on both sets.

In [ ]:
# Replace every ______ below, then run this cell.
rows = []

for p in penalties:
    lasso_pipe.set_params(model__alpha=p)
    lasso_pipe.fit(______[predictors], ______["glucose"])

    fit_rmse = root_mean_squared_error(fitting["glucose"], lasso_pipe.predict(fitting[predictors]))
    val_rmse = root_mean_squared_error(______["glucose"], lasso_pipe.predict(______[predictors]))
    rows.append({"penalty": p, "fit_rmse": fit_rmse, "val_rmse": val_rmse})

val_results = pd.DataFrame(rows)
val_results.round(2)

> **Stuck?**
>
> Each candidate is fitted on the 450 `fitting` rows and scored on the 150 rows
> it has not seen.

In [ ]:
fig, ax = plt.subplots()
ax.plot(val_results["penalty"], val_results["fit_rmse"], marker="o", label="fit_rmse")
ax.plot(val_results["penalty"], val_results["val_rmse"], marker="o", label="val_rmse")
ax.set_xscale("log")
ax.set_xlabel("Penalty (log scale)")
ax.set_ylabel("RMSE (mg/dL)")
ax.set_title("The lasso: fitting and validation RMSE for each penalty")
ax.legend()
plt.show()

The fitting RMSE rises with the penalty, as in Part 2. The validation RMSE
**falls**, from 25.16 to 24.55 at a penalty of 0.1, and then rises again, to
29.87 at 10. With a small penalty the model learns noise in the fitting rows
that does not repeat in the validation rows (**overfitting**); with a large
penalty it misses the real pattern too (**underfitting**). The validation rows
show both, so we choose the penalty with the lowest validation RMSE: 0.1. This
is the normal way to choose a hyperparameter.

------------------------------------------------------------------------

### Part 4 — The same search with `GridSearchCV()`

scikit-learn runs this loop for you. `GridSearchCV()` takes the pipeline, the
grid of candidates, and the split to score them on. `PredefinedSplit()` gives
it our one split: training rows marked `-1` are used for fitting, rows marked
`0` are the validation rows. `GridSearchCV()` always looks for the **highest**
score, so for an error it uses the negative RMSE.

In [ ]:
is_val = train.index.isin(validation.index)
one_split = PredefinedSplit(np.where(is_val, 0, -1))   # 0 = validation row, -1 = fitting row

lasso_search = GridSearchCV(lasso_pipe,
                            param_grid={"model__alpha": penalties},
                            cv=one_split,
                            scoring="neg_root_mean_squared_error")
lasso_search.fit(train[predictors], train["glucose"])

results = pd.DataFrame(lasso_search.cv_results_)
results["val_rmse"] = -results["mean_test_score"]      # back to a positive RMSE
print(results[["param_model__alpha", "val_rmse"]].round(2))
print(lasso_search.best_params_)

The `val_rmse` column is the validation RMSE of your loop, number for number,
and `best_params_` is the same penalty, 0.1. (scikit-learn calls the held-out
rows "test" in its names, such as `mean_test_score`; here they are our
validation rows.) `ValidationCurveDisplay` draws the fitting and the validation
error for each value:

In [ ]:
display = ValidationCurveDisplay.from_estimator(
    lasso_pipe, train[predictors], train["glucose"],
    param_name="model__alpha", param_range=penalties,
    cv=one_split, scoring="neg_root_mean_squared_error", negate_score=True,
    std_display_style=None)
display.ax_.set_xscale("log")
display.ax_.legend(["fitting rows", "validation rows"])
display.ax_.set_title("ValidationCurveDisplay: RMSE for each penalty")
plt.show()

| Your loop in Part 3 | `GridSearchCV()` |
|:--------------------|:-----------------|
| `for p in penalties:` | `param_grid={"model__alpha": penalties}` |
| `fit(fitting[...])` | the rows marked `-1` in `one_split` |
| score on `validation` with `root_mean_squared_error` | the rows marked `0`, `scoring="neg_root_mean_squared_error"` |
| read the table for the lowest `val_rmse` | `best_params_` |

> **Where to find a model's hyperparameters**
>
> `get_params()` lists every setting of every step of a pipeline, under the names
> that `set_params()` and `param_grid` use. The
> [scikit-learn page for `ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)
> explains `alpha` and `l1_ratio`, and the user guide on
> [tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)
> covers the search tools.

In [ ]:
[name for name in lasso_pipe.get_params() if name.startswith("model__")]

------------------------------------------------------------------------

### Part 5 — Tuning the mix of lasso and ridge

`ElasticNet()` has a second hyperparameter, `l1_ratio`: 1 is the lasso, 0 is
ridge, and values in between mix the two penalties. Keep the penalty chosen in
Part 4 and tune only `l1_ratio`:

In [ ]:
# Replace every ______ below, then run this cell.
best_alpha = lasso_search.best_params_["model__alpha"]   # the penalty chosen in Part 4

mix_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("model", ElasticNet(alpha=best_alpha, max_iter=100000)),
])

mix_search = GridSearchCV(mix_pipe,
                          param_grid={"model__l1_ratio": [0, 0.25, 0.5, 0.75, ______]},
                          cv=one_split,
                          scoring="neg_root_mean_squared_error")
mix_search.fit(train[predictors], train["glucose"])

mix_results = pd.DataFrame(mix_search.cv_results_)
mix_results["val_rmse"] = -mix_results["mean_test_score"]
print(mix_results[["param_model__l1_ratio", "val_rmse"]].round(2))
print(mix_search.best_params_)

> **Stuck?**
>
> `l1_ratio` runs from 0 (ridge) to 1 (lasso).

------------------------------------------------------------------------

### Part 6 — Refitting the chosen model, and its one test

Every candidate was fitted on the 450 fitting rows only. The chosen one is now
fitted once more on all 600 training rows, with the same settings: the
**refit**. Do it by hand, then with the tool, and check that they agree.

**By hand.** Take the row with the lowest validation RMSE, put its value into a
fresh copy of the pipeline with `set_params()`, and fit. `clone()` makes an
unfitted copy of the pipeline, so the one inside the search is left alone.

In [ ]:
# Replace every ______ below, then run this cell.
from sklearn.base import clone

best = mix_results.loc[mix_results["val_rmse"].idxmin(), "param_model__l1_ratio"]   # the lowest validation RMSE
print("best l1_ratio:", best)

refit_by_hand = clone(mix_pipe).set_params(model__l1_ratio=best)   # put the chosen value in
refit_by_hand.fit(______[predictors], ______["glucose"])           # all 600 training rows

> **Stuck?**
>
> The refit uses every training row, fitting and validation together.

**With the tool.** `GridSearchCV()` has already done the refit: after the
search it puts the best value into the pipeline and fits it on all the rows it
was given (`refit=True`, the default). The result is in `best_estimator_`.

In [ ]:
refit_by_tool = mix_search.best_estimator_

Are they the same model? Put the coefficients side by side:

In [ ]:
coefs = pd.DataFrame({"by_hand": refit_by_hand["model"].coef_,
                      "best_estimator_": refit_by_tool["model"].coef_},
                     index=refit_by_tool[:-1].get_feature_names_out())   # the 35 column names
print(coefs.head().round(3))
print("same model:", np.allclose(coefs["by_hand"], coefs["best_estimator_"]))

`True`: every coefficient is the same. `best_estimator_` is the refit you did
by hand.

Now the **one** test check, with the unpenalised model for comparison:

In [ ]:
plain_pipe = Pipeline(mix_pipe.steps[:-1] + [("model", LinearRegression())])
plain_pipe.fit(train[predictors], train["glucose"])

print("chosen model, test RMSE:     ", round(root_mean_squared_error(test["glucose"], refit_by_tool.predict(test[predictors])), 2))
print("unpenalised model, test RMSE:", round(root_mean_squared_error(test["glucose"], plain_pipe.predict(test[predictors])), 2))
print("columns kept:", (refit_by_tool["model"].coef_ != 0).sum())

The chosen model (penalty 0.1, `l1_ratio` 1: the lasso) has a test RMSE of
27.29 against 27.14 for the unpenalised model: about the same, with 23 of the
35 columns.

**Question!** The penalised model is about as accurate on the test rows as the
unpenalised one, but it uses fewer columns. Why might that still matter?

> **From one validation set to cross-validation**
>
> One validation set depends on which 150 people landed in it (try the Stretch).
> Exercise 5.4 introduces **cross-validation**, which repeats the comparison on
> several splits and averages. In the code only `cv` changes (`cv=5` instead of
> `one_split`); `best_params_` and `best_estimator_` work as here.
>
> ```python
> GridSearchCV(lasso_pipe, param_grid=..., cv=one_split, ...)   # this exercise
> GridSearchCV(lasso_pipe, param_grid=..., cv=5, ...)           # Exercise 5.4
> ```

> **Key points**
>
> - A hyperparameter is chosen, not fitted: try several values, score each on rows the model was not fitted on, and keep the best.
> - Scored on its own training rows, the least restrained model always wins. That is why we score on validation rows.
> - `GridSearchCV()` is the loop of Part 3 written for you; `best_params_` is the winner and `best_estimator_` is the refit.
> - Refit the chosen model on all the training rows, then check it once on the test rows.

> **Other ways to choose the candidates**
>
> A regular grid is the simplest way to choose which values to try, but not the
> only one. **Random search** draws the candidates at random from a range
> (`grid_random()` in tidymodels, `RandomizedSearchCV()` in scikit-learn).
> **Space-filling designs**, such as the **Latin hypercube**, spread a fixed
> number of candidates evenly over the range
> ([`grid_space_filling()`](https://dials.tidymodels.org/reference/grid_space_filling.html)).
> **Iterative** searches, such as Bayesian optimisation, choose each new
> candidate from the results so far. All of them score the candidates the same
> way as here. Read more in chapters 13 and 14 of Tidy Modeling with R
> ([grid search](https://www.tmwr.org/grid-search),
> [iterative search](https://www.tmwr.org/iterative-search)) and in the
> scikit-learn guide to
> [tuning hyper-parameters](https://scikit-learn.org/stable/modules/grid_search.html).

## Stretch

> **Stretch — a different validation set**
>
> Draw a different validation set by changing the seed of the second split
> (`set.seed(1)` before `initial_split(train, prop = 0.75)` in R,
> `random_state=1` in the second `train_test_split()` in Python) and run the
> search of Part 3 or Part 4 again. Is the same penalty
> chosen? What does that tell you about choosing with a single validation set?